# **CST8508 Machine Vision - Lab 5: Cats vs Dogs Image Classification**

**Objective:** Implement an end-to-end Convolutional Neural Network (CNN) to classify images from the Cats vs. Dogs dataset. You can download the dataset from here https://www.microsoft.com/en-us/download/details.aspx?id=54765
Do not forget to change the runtime type of your notebook to GPU so you can train on a GPU.

**Lab Instructions:**

* Implement each function in a modular fashion.
* Ensure functions interact with each other seamlessly.
* Document each step with comments for clarity.
* After implementing all parts, run the entire pipeline on the Cats vs. Dogs dataset and analyze the results.


This lab will provide a comprehensive understanding of building and training a CNN for image classification, from data preprocessing to model evaluation.







In [2]:
# ── Dataset Setup ──────────────────────────────────────────────────────────────
# 自动下载并解压 Microsoft Cats vs Dogs 数据集，同时清除损坏图片
# Auto-download Microsoft Cats vs Dogs dataset and remove corrupted images

import os, zipfile, urllib.request
from pathlib import Path
from PIL import Image

DATASET_URL  = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
ZIP_PATH     = Path("kagglecatsanddogs_5340.zip")
DATASET_PATH = Path("PetImages")   # ← load_dataset 使用此路径

# 1. 下载 / Download (~786 MB)
if not ZIP_PATH.exists() and not DATASET_PATH.exists():
    print("Downloading dataset (~786 MB) ...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print("Download complete.")
else:
    print("Zip / dataset already present, skipping download.")

# 2. 解压 / Extract
if not DATASET_PATH.exists():
    print("Extracting ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Extraction complete.")

# 3. 清理损坏图片（此数据集存在部分无效 JPEG）
# Remove corrupted images (known issue with this dataset)
removed = 0
for img_path in DATASET_PATH.rglob("*.jpg"):
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        img_path.unlink()
        removed += 1
print(f"Removed {removed} corrupted images.")
print(f"Dataset ready at: {DATASET_PATH.resolve()}")


Zip / dataset already present, skipping download.


c:\Users\40270\OneDrive\Desktop\workspace\aisd\.venv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Removed 0 corrupted images.
Dataset ready at: C:\Users\40270\OneDrive\Desktop\workspace\aisd\courses\mv\code\lab5\PetImages


**Part 1:** Data Loading and Augmentation

**Function load_dataset(path):** Load the Cats vs. Dogs dataset from the given path. Split into training and test sets. Define transforms for augmentation and normalization.


In [3]:
import os, random
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def load_dataset(path, split_ratio=0.8):
    # Load images from path
    train_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    test_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Split into training and test sets
    full = datasets.ImageFolder(root=path, transform=train_transform)
    n = len(full)
    train_n = int(n * split_ratio)
    idx = list(range(n))
    random.shuffle(idx)
    train_data = Subset(full, idx[:train_n])
    test_data  = Subset(datasets.ImageFolder(root=path, transform=test_transform), idx[train_n:])

    # Normalize pixel values (handled inside transforms above)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=2)
    test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False, num_workers=2)
    print(f"Train: {train_n}  Test: {n - train_n}  Classes: {full.classes}")
    return train_loader, test_loader


Using device: cuda


**Part 2:** Model Definition

**Function define_model():** Define a CNN model with layers (Conv2D, MaxPooling, Flatten, Dense). Include activation functions and the optimizer.

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Conv2D → MaxPooling blocks
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        # Dense layers after Flatten
        self.fc1     = nn.Linear(128 * 16 * 16, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(256, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 128→64
        x = self.pool(F.relu(self.conv2(x)))   # 64→32
        x = self.pool(F.relu(self.conv3(x)))   # 32→16
        x = x.view(x.size(0), -1)              # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def define_model():
    model = SimpleCNN().to(DEVICE)
    return model


**Part 3:** Model Training

**Function train_model(model, train_data, validation_data):** Train the model using the training set with validation data. Set epochs and batch size.


In [5]:
import torch.optim as optim

def train_model(model, train_loader, test_loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    # Train the model using fit()
    for epoch in range(1, epochs + 1):
        # Set epochs, batch size
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss /= total
        train_acc = correct / total

        # Use validation data
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                val_loss += criterion(out, labels).item() * imgs.size(0)
                val_correct += (out.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch [{epoch:02d}/{epochs}]  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    return history


**Part 4:** Model Evaluation

**Function evaluate_and_predict
(model, test_loader):** Evaluate the model's performance on the test dataset. Return accuracy

In [6]:
def evaluate_and_predict(model, test_loader):
    model.eval()
    predictions, actual_labels = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.tolist())

    accuracy = sum(p == a for p, a in zip(predictions, actual_labels)) / len(actual_labels)
    print(f"Accuracy: {accuracy:.4f}")

    from sklearn.metrics import classification_report
    print(classification_report(actual_labels, predictions, target_names=['Cat', 'Dog']))

    return accuracy, predictions, actual_labels


**Running the code**

In [7]:
train_loader, test_loader = load_dataset(str(DATASET_PATH))
model = define_model()
train_model(model, train_loader, test_loader, epochs=10)
accuracy, predictions, actual_labels = evaluate_and_predict(model, test_loader)


Train: 19998  Test: 5000  Classes: ['Cat', 'Dog']
Epoch [01/10]  train_loss=0.6392  train_acc=0.6245  val_loss=0.5441  val_acc=0.7272
Epoch [02/10]  train_loss=0.5251  train_acc=0.7399  val_loss=0.4814  val_acc=0.7734
Epoch [03/10]  train_loss=0.4643  train_acc=0.7849  val_loss=0.4534  val_acc=0.7886
Epoch [04/10]  train_loss=0.4172  train_acc=0.8099  val_loss=0.3730  val_acc=0.8368
Epoch [05/10]  train_loss=0.3869  train_acc=0.8283  val_loss=0.3536  val_acc=0.8498
Epoch [06/10]  train_loss=0.3607  train_acc=0.8413  val_loss=0.3382  val_acc=0.8592
Epoch [07/10]  train_loss=0.3387  train_acc=0.8548  val_loss=0.3050  val_acc=0.8736
Epoch [08/10]  train_loss=0.3162  train_acc=0.8613  val_loss=0.2999  val_acc=0.8742
Epoch [09/10]  train_loss=0.3018  train_acc=0.8717  val_loss=0.2863  val_acc=0.8770
Epoch [10/10]  train_loss=0.2863  train_acc=0.8809  val_loss=0.2727  val_acc=0.8842
Accuracy: 0.8842
              precision    recall  f1-score   support

         Cat       0.90      0.86     